**Data Cleaning**

In [82]:
import ast
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MultiLabelBinarizer
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import os
import ast
import warnings
warnings.filterwarnings('ignore')
import re
from collections import Counter
pd.set_option('display.max_columns', None)
DATA_DIR = 'data'

In [83]:
os.listdir(DATA_DIR)

['goodreads-reviews.csv',
 'rotten-tomatoes-audience-reviews.csv',
 'cleaned',
 'rotten-tomatoes-critic-reviews.csv',
 'combined_letterboxd_tables.csv',
 'goodreads-books.csv',
 'goodreads-listings-cache.json',
 'letterboxd-full_top500movies.csv',
 'rotten-tomatoes-movies.csv']

In [ ]:
# Load all datasets from CSV
goodreads_books = pd.read_csv(DATA_DIR + '/goodreads-books.csv')
goodreads_reviews = pd.read_csv(DATA_DIR + '/goodreads-reviews.csv')
rotten_tomatoes_movies = pd.read_csv(DATA_DIR + '/rotten-tomatoes-movies.csv')
rotten_tomatoes_audience_reviews = pd.read_csv(DATA_DIR + '/rotten-tomatoes-audience-reviews.csv')
rotten_tomatoes_critic_reviews = pd.read_csv(DATA_DIR + '/rotten-tomatoes-critic-reviews.csv')

In [ ]:
sum_rows = 0
for name, df in [("Goodreads Books", goodreads_books), ("Goodreads Reviews", goodreads_reviews),
                 ("RT Movies", rotten_tomatoes_movies), ("RT Audience Reviews", rotten_tomatoes_audience_reviews),
                 ("RT Critic Reviews", rotten_tomatoes_critic_reviews)]:
    sum_rows += df.shape[0]
    print(f"{name:30s} {str(df.shape):>15s}")

Goodreads Books                     (6330, 32)
Goodreads Reviews                 (177261, 11)
RT Movies                           (3001, 48)
RT Audience Reviews                 (15988, 6)
RT Critic Reviews                   (25581, 7)
Letterboxd Movies                   (1000, 13)


In [86]:
print(f"Total rows of data: {sum_rows}")

Total rows of data: 229161


## Data Cleaning
We work on copies of every dataframe to preserve the raw originals for comparison. For the copied dataframes we did four different steps:

1. **Duplicate removal** - drops identical rows across all datasets.
2. **Column selection** - retains only fields needed for analysis; drops URLs, internal IDs, and platform-specific metadata.
3. **Missing-value handling** - drops rows with no usable key (book_id, avg_rating); fills categorical NaNs with `'Unknown'`; fills counts with 0.
4. **Text cleaning** - strips HTML, URLs, and encoding artifacts; collapses whitespace; removes reviews with fewer than 5 words.

In [ ]:
# Work on copies so we can always compare back to the originals
cleaned_goodreads_books = goodreads_books.copy()
cleaned_goodreads_reviews = goodreads_reviews.copy()
cleaned_rotten_tomatoes_movies = rotten_tomatoes_movies.copy()
cleaned_rotten_tomatoes_audience_reviews = rotten_tomatoes_audience_reviews.copy()
cleaned_rotten_tomatoes_critic_reviews = rotten_tomatoes_critic_reviews.copy()

In [ ]:
for name, df in [("Goodreads Books", cleaned_goodreads_books),
                 ("Goodreads Reviews", cleaned_goodreads_reviews),
                 ("RT Movies", cleaned_rotten_tomatoes_movies),
                 ("RT Audience Reviews", cleaned_rotten_tomatoes_audience_reviews),
                 ("RT Critic Reviews", cleaned_rotten_tomatoes_critic_reviews)]:
    before = len(df)
    df.drop_duplicates(inplace=True)
    after = len(df)
    if before != after:
        print(f"  {name}: {before:,} -> {after:,}  (dropped {before - after:,} duplicates)")

print("  Duplicate removal complete.")

  Goodreads Books: 6,330 -> 5,788  (dropped 542 duplicates)
  Goodreads Reviews: 177,261 -> 144,142  (dropped 33,119 duplicates)
  RT Audience Reviews: 15,988 -> 15,986  (dropped 2 duplicates)
  RT Critic Reviews: 25,581 -> 25,578  (dropped 3 duplicates)
  Duplicate removal complete.


In [ ]:
# Goodreads Books — keep core metadata, genres, description, ratings
cleaned_goodreads_books.drop(columns=[
    'url', 'legacy_id', 'image_url', 'author_url',
    'num_pages', 'book_format', 'publisher',
    'language', 'series_name', 'series_position', 'content_rating',
    'isbn13', 'asin', 'isbn', 'awards', 'source_list', 'list_rank'
], inplace=True, errors='ignore')

# Goodreads Reviews — keep title, reviewer_id, rating, review_text
cleaned_goodreads_reviews.drop(columns=[
    'book_url', 'reviewer_name', 'created_at', 'updated_at',
    'like_count', 'comment_count', 'shelves'
], inplace=True, errors='ignore')

# RT Movies — keep title, genres, scores, consensus text, synopsis
cleaned_rotten_tomatoes_movies.drop(columns=[
    'url', 'critic_score_browse', 'content_rating', 'audience_score_browse', 'name',
    'critic_certified', 'critic_liked', 'critic_not_liked', 'runtime',
    'top_critic_score', 'top_critic_count', 'top_critic_liked', 'top_critic_not_liked',
    'top_critic_sentiment', 'top_critic_certified',
    'actors', 'directors', 'release_date',
    'audience_certified', 'audience_liked', 'audience_not_liked', 'audience_rating_band',
    'audience_score_type', 'audience_all_liked', 'audience_all_not_liked', 'box_office_usa',
    'original_language', 'release_date_theaters', 'release_date_streaming',
    'distributor', 'production_co'
], inplace=True, errors='ignore')

# RT Audience Reviews — keep title, score, review_text
cleaned_rotten_tomatoes_audience_reviews.drop(columns=['movie_url', 'user_name', 'verified'], inplace=True, errors='ignore')

# RT Critic Reviews — keep title, rating, review_text, sentiment
cleaned_rotten_tomatoes_critic_reviews.drop(columns=['movie_url', 'critic_name', 'publication'], inplace=True, errors='ignore')

print("Columns dropped. Remaining columns per dataset:")
for name, df in [("Goodreads Books", cleaned_goodreads_books), ("Goodreads Reviews", cleaned_goodreads_reviews),
                 ("RT Movies", cleaned_rotten_tomatoes_movies), ("RT Audience Reviews", cleaned_rotten_tomatoes_audience_reviews),
                 ("RT Critic Reviews", cleaned_rotten_tomatoes_critic_reviews)]:
    print(f"  {name}: {list(df.columns)}")


Columns dropped. Remaining columns per dataset:
  Goodreads Books: ['title', 'title_complete', 'book_id', 'description', 'author', 'avg_rating', 'ratings_count', 'reviews_count', 'rating_dist_1', 'rating_dist_2', 'rating_dist_3', 'rating_dist_4', 'rating_dist_5', 'publication_date', 'genres']
  Goodreads Reviews: ['book_title', 'reviewer_id', 'rating', 'review_text']
  RT Movies: ['title', 'genres', 'tomatometer', 'tomatometer_count', 'critic_avg_rating', 'critic_sentiment', 'audience_score', 'audience_avg_rating', 'audience_review_count', 'audience_sentiment', 'audience_all_score', 'audience_all_avg_rating', 'audience_all_review_count', 'audience_all_sentiment', 'synopsis', 'critics_consensus', 'audience_consensus']
  RT Audience Reviews: ['movie_title', 'score', 'review_text']
  RT Critic Reviews: ['movie_title', 'sentiment', 'rating', 'review_text']
  Letterboxd: ['Rank', 'Name', 'Year', 'Director', 'Rating (Out of 5)', 'Genres', 'Tier']


In [90]:
# ── Goodreads Books ──

# Drop rows with no book_id or no avg_rating (can't use them)
cleaned_goodreads_books = cleaned_goodreads_books.dropna(subset=['book_id', 'avg_rating'])

# Fill title_complete with title where missing, then strip whitespace
cleaned_goodreads_books['title_complete'] = cleaned_goodreads_books['title_complete'].fillna(cleaned_goodreads_books['title'])
cleaned_goodreads_books['title_complete'] = cleaned_goodreads_books['title_complete'].str.strip()
cleaned_goodreads_books['title'] = cleaned_goodreads_books['title'].str.strip()

# Fill missing metadata with 'Unknown', missing counts with 0
cleaned_goodreads_books[['description', 'author', 'genres', 'publication_date']] = (
    cleaned_goodreads_books[['description', 'author', 'genres', 'publication_date']].fillna('Unknown')
)
cleaned_goodreads_books['reviews_count'] = cleaned_goodreads_books['reviews_count'].fillna(0)


# ── Goodreads Reviews ──

# Only keep reviews for books that survived cleaning
valid_titles = cleaned_goodreads_books['title'].unique()
cleaned_goodreads_reviews = cleaned_goodreads_reviews[
    cleaned_goodreads_reviews['book_title'].isin(valid_titles)
].copy()


# Fill empty review text (prevents NLP crashes), keep rating NaNs as-is
cleaned_goodreads_reviews['review_text'] = cleaned_goodreads_reviews['review_text'].fillna("")
cleaned_goodreads_reviews['rating'] = cleaned_goodreads_reviews['rating'].astype('Int64')


# ── RT Movies ──

cleaned_rotten_tomatoes_movies['title'] = cleaned_rotten_tomatoes_movies['title'].str.strip()
cleaned_rotten_tomatoes_movies[['critics_consensus', 'audience_consensus']] = (
    cleaned_rotten_tomatoes_movies[['critics_consensus', 'audience_consensus']].fillna("")
)
cleaned_rotten_tomatoes_movies[['genres', 'synopsis']] = (
    cleaned_rotten_tomatoes_movies[['genres', 'synopsis']].fillna('Unknown')
)

print("NaN handling complete.")

NaN handling complete.


In [91]:
# dropping rows without usable review content

# For Q1 ONLY we need either a rating or review text - rows with neither are useless
for label, df, text_col, rating_col in [
    ("Goodreads Reviews",   cleaned_goodreads_reviews, 'review_text', 'rating'),
    ("RT Audience Reviews",  cleaned_rotten_tomatoes_audience_reviews, 'review_text', 'score'),
    ("RT Critic Reviews",    cleaned_rotten_tomatoes_critic_reviews, 'review_text', 'rating'),
]:
    before = len(df)
    if label == "Goodreads Reviews":
        mask = df[rating_col].notna() | (df[text_col].str.strip() != "")
        cleaned_goodreads_reviews = df[mask]
    elif label == "RT Audience Reviews":
        mask = df[text_col].str.strip() != ""
        cleaned_rotten_tomatoes_audience_reviews = df[mask]
    elif label == "RT Critic Reviews":
        mask = df[text_col].notna() & (df[text_col].str.strip() != "")
        cleaned_rotten_tomatoes_critic_reviews = df[mask]
    after = len(mask[mask].index)
    print(f"{label}: {before:,} → {after:,}  (dropped {before - after:,})")


Goodreads Reviews: 144,142 → 144,131  (dropped 11)
RT Audience Reviews: 15,986 → 15,986  (dropped 0)
RT Critic Reviews: 25,578 → 25,436  (dropped 142)


In [98]:
cleaned_goodreads_reviews.head().iloc[1]["review_text"][:500]

'One Million Stars! Reread#3 ✔️ "Speak again the ancient oath: Life before death. Strength before weakness. Journey before destination. And return to men the Shards they once bore. The knight radiants must stand again." There is a reason why this series is the highest rated series on goodreads. It had everything: •Epic battles (a lot of them) •intense, gripping scenes •Chosen one •Found family •Amazing world building •Big plot twist •Unique magic systems •KALADIN FREAKING STORMBLESSED Now lemme s'

In [99]:
# cleaning text for analysis
def clean_review_text(text):
    """Strip HTML, URLs, encoding artifacts, and normalize whitespace."""
    if not isinstance(text, str) or text.strip() == "":
        return ""
    text = re.sub(r'<[^>]+>', ' ', text)                    # HTML tags
    text = re.sub(r'https?://\S+|www\.\S+', '', text)      # URLs
    text = (text.replace('\u2019', "'").replace('\u201c', '"').replace('\u201d', '"')
                .replace('\u2014', '—').replace('\u2013', '–').replace('\u2026', '...')
                .replace('\xa0', ' '))
    text = text.replace('â€™', "'").replace('â€œ', '"').replace('â€\x9d', '"')
    text = text.replace('â€"', '—').replace('â€"', '–').replace('â€¦', '...')
    text = text.replace('Â', '')
    text = re.sub(r'[^\x00-\x7F]+', ' ', text)              # remaining non-ASCII
    text = re.sub(r'\s+', ' ', text).strip()                 # collapse whitespace
    return text

# Apply to all review text columns
cleaned_goodreads_reviews['review_text'] = cleaned_goodreads_reviews['review_text'].apply(clean_review_text)
cleaned_rotten_tomatoes_audience_reviews['review_text'] = cleaned_rotten_tomatoes_audience_reviews['review_text'].apply(clean_review_text)
cleaned_rotten_tomatoes_critic_reviews['review_text'] = cleaned_rotten_tomatoes_critic_reviews['review_text'].apply(clean_review_text)
cleaned_rotten_tomatoes_movies['critics_consensus'] = cleaned_rotten_tomatoes_movies['critics_consensus'].apply(clean_review_text)
cleaned_rotten_tomatoes_movies['audience_consensus'] = cleaned_rotten_tomatoes_movies['audience_consensus'].apply(clean_review_text)

# Sanity check
sample = cleaned_goodreads_reviews.head().iloc[1]
print("   Sample cleaned review:")
print(f"  Rating: {sample['rating']}")
print(f"  Text:   {sample['review_text'][:500]}...")


   Sample cleaned review:
  Rating: 5
  Text:   One Million Stars! Reread#3 "Speak again the ancient oath: Life before death. Strength before weakness. Journey before destination. And return to men the Shards they once bore. The knight radiants must stand again." There is a reason why this series is the highest rated series on goodreads. It had everything: Epic battles (a lot of them) intense, gripping scenes Chosen one Found family Amazing world building Big plot twist Unique magic systems KALADIN FREAKING STORMBLESSED Now lemme start with t...


In [100]:
# filtering out trivial short reviews since this would just contirbute to the noise
MIN_WORDS = 5

# Add word counts
for df in [cleaned_goodreads_reviews, cleaned_rotten_tomatoes_audience_reviews, cleaned_rotten_tomatoes_critic_reviews]:
    df['word_count'] = df['review_text'].str.split().str.len().fillna(0).astype(int)

print("Word-count stats BEFORE filtering:")
for label, df in [("Goodreads", cleaned_goodreads_reviews),
                  ("RT Audience", cleaned_rotten_tomatoes_audience_reviews),
                  ("RT Critics", cleaned_rotten_tomatoes_critic_reviews)]:
    short = (df['word_count'] < MIN_WORDS).sum()
    print(f"  {label:14s}  median={df['word_count'].median():.0f}  mean={df['word_count'].mean():.1f}  "
          f"< {MIN_WORDS} words: {short:,}")

# Filter
cleaned_goodreads_reviews = cleaned_goodreads_reviews[cleaned_goodreads_reviews['word_count'] >= MIN_WORDS]
cleaned_rotten_tomatoes_audience_reviews = cleaned_rotten_tomatoes_audience_reviews[
    cleaned_rotten_tomatoes_audience_reviews['word_count'] >= MIN_WORDS]
cleaned_rotten_tomatoes_critic_reviews = cleaned_rotten_tomatoes_critic_reviews[
    cleaned_rotten_tomatoes_critic_reviews['word_count'] >= MIN_WORDS]

print(f"\nAfter filtering (≥ {MIN_WORDS} words):")
print(f"  Goodreads reviews:   {len(cleaned_goodreads_reviews):,}")
print(f"  RT audience reviews: {len(cleaned_rotten_tomatoes_audience_reviews):,}")
print(f"  RT critic reviews:   {len(cleaned_rotten_tomatoes_critic_reviews):,}")

Word-count stats BEFORE filtering:
  Goodreads       median=188  mean=305.0  < 5 words: 3,555
  RT Audience     median=24  mean=44.8  < 5 words: 720
  RT Critics      median=25  mean=24.7  < 5 words: 299

After filtering (≥ 5 words):
  Goodreads reviews:   140,576
  RT audience reviews: 15,266
  RT critic reviews:   25,137


In [101]:
def parse_critic_rating(raw):
    """
    Parse RT critic ratings from mixed formats into a 0–5 float scale.

    Handles:
      Fractions with spaces:   '3.5 / 5', '3/ 5', '2. 5 / 5'
      Fractions:               '3/5', '7/10', '75/100', '18/20', '.5/4'
      Percentages:             '75%', '70%'
      Letter grades:           'A-', 'B+', 'A-minus', 'B-plus', 'C-plus', 'C +', 'A minus'
      Word-number stars:       'FIVE STARS', 'THREE STARS'
      Numeric stars:           '3 stars', '1 star', '.5 stars'
      Mixed fraction stars:    '3 1/2 stars', '3 1/4 stars'
      Mixed fraction w/ scale: '3 1/2/5'
      "X out of Y" / "X of Y": '2 out of 5', '4.5 of 5', '3.5 out of 5'
      Plain numeric:           '4', '7.5', '55', '9.5 '
      Qualitative (mapped):    'Recommended', 'Must Watch', 'Catch It On Cable', etc.
    Returns NaN if unparseable.
    """
    if pd.isna(raw):
        return np.nan
    raw = str(raw).strip()
    if raw == '':
        return np.nan

    # ── Qualitative / text-only ratings → approximate numeric ──
    qual_map = {
        'must watch': 5.0,
        'highly recommended': 4.5,
        'recommended': 4.0,
        'catch it on cable': 2.5,
        'half price': 2.5,
    }
    if raw.lower() in qual_map:
        return qual_map[raw.lower()]

    # ── "4 Ripples" or similar custom scale labels ──
    ripple = re.match(r'^([\d.]+)\s*ripples?$', raw, re.IGNORECASE)
    if ripple:
        return round(float(ripple.group(1)), 2)

    # ── "C=" typo → treat as "C" ──
    cleaned = raw.replace('=', '').strip()

    # ── Letter grades (including spelled-out: 'A-minus', 'B-plus', 'C +', 'A minus') ──
    grade_str = cleaned
    grade_str = re.sub(r'[\s-]*minus$', '-', grade_str, flags=re.IGNORECASE)
    grade_str = re.sub(r'[\s-]*plus$', '+', grade_str, flags=re.IGNORECASE)
    grade_str = re.sub(r'\s+', '', grade_str)  # collapse spaces like "C +"

    letter_grades = {
        'A+': 5.0, 'A': 4.7, 'A-': 4.3,
        'B+': 4.0, 'B': 3.7, 'B-': 3.3,
        'C+': 3.0, 'C': 2.7, 'C-': 2.3,
        'D+': 2.0, 'D': 1.7, 'D-': 1.3,
        'F+': 1.0, 'F': 0.5, 'F-': 0.0,
    }
    if grade_str.upper() in letter_grades:
        return letter_grades[grade_str.upper()]

    # ── Word-number stars: "FIVE STARS", "THREE STARS" ──
    word_nums = {
        'zero': 0, 'one': 1, 'two': 2, 'three': 3, 'four': 4, 'five': 5, 'half': 0.5,
    }
    word_star = re.match(r'^(\w+)\s+stars?$', raw, re.IGNORECASE)
    if word_star and word_star.group(1).lower() in word_nums:
        return float(word_nums[word_star.group(1).lower()])

    # ── Numeric stars: "3 stars", "1 star", ".5 stars" ──
    num_star = re.match(r'^([\d.]+)\s+stars?$', raw, re.IGNORECASE)
    if num_star:
        return round(float(num_star.group(1)), 2)

    # ── Mixed fraction stars: "3 1/2 stars", "3 1/4 stars" ──
    mixed_star = re.match(r'^(\d+)\s+(\d+)/(\d+)\s+stars?$', raw, re.IGNORECASE)
    if mixed_star:
        whole = int(mixed_star.group(1))
        frac = int(mixed_star.group(2)) / int(mixed_star.group(3))
        return round(whole + frac, 2)

    # ── Mixed fraction with scale denominator: "3 1/2/5" → (3.5 / 5) * 5 ──
    mixed_frac = re.match(r'^(\d+)\s+(\d+)/(\d+)/(\d+)$', raw.strip())
    if mixed_frac:
        whole = int(mixed_frac.group(1))
        frac = int(mixed_frac.group(2)) / int(mixed_frac.group(3))
        scale = int(mixed_frac.group(4))
        return round(((whole + frac) / scale) * 5, 2) if scale > 0 else np.nan

    # ── "X out of Y" / "X of Y" ──
    out_of = re.match(r'^([\d.]+)\s+(?:out\s+)?of\s+([\d.]+)$', raw, re.IGNORECASE)
    if out_of:
        num, denom = float(out_of.group(1)), float(out_of.group(2))
        return round((num / denom) * 5, 2) if denom > 0 else np.nan

    # ── Percentage: "75%" ──
    pct = re.match(r'^([\d.]+)\s*%$', raw)
    if pct:
        return round(float(pct.group(1)) / 100 * 5, 2)

    # ── Fraction (with possible spaces): "3.5 / 5", "3/ 5", "2. 5 / 5" ──
    norm = re.sub(r'(\d+\.)\s+(\d+)', r'\1\2', raw.strip())  # "2. 5" → "2.5"
    fraction_match = re.match(r'^([\d.]+)\s*/\s*([\d.]+)$', norm)
    if fraction_match:
        num, denom = float(fraction_match.group(1)), float(fraction_match.group(2))
        return round((num / denom) * 5, 2) if denom > 0 else np.nan

    # ── Plain numeric: "4", "7.5", "55", "9.5 " ──
    try:
        val = float(raw)
        if val <= 5:    return round(val, 2)
        elif val <= 10: return round((val / 10) * 5, 2)
        elif val <= 100: return round((val / 100) * 5, 2)
    except ValueError:
        pass

    return np.nan


# Parse critic ratings to 0-5
cleaned_rotten_tomatoes_critic_reviews['rating_parsed'] = (
    cleaned_rotten_tomatoes_critic_reviews['rating'].apply(parse_critic_rating)
)

parsed = cleaned_rotten_tomatoes_critic_reviews['rating_parsed'].notna().sum()
total = len(cleaned_rotten_tomatoes_critic_reviews)
originally_null = cleaned_rotten_tomatoes_critic_reviews['rating'].isna().sum()
still_unparsed = cleaned_rotten_tomatoes_critic_reviews[
    cleaned_rotten_tomatoes_critic_reviews['rating'].notna() &
    cleaned_rotten_tomatoes_critic_reviews['rating_parsed'].isna()
]
print(f"RT critic ratings: {total} total, {originally_null} originally null")
print(f"Parsed successfully: {parsed}/{total - originally_null} non-null ratings ({parsed/(total-originally_null)*100:.1f}%)")
if len(still_unparsed) > 0:
    print(f"Could not parse {len(still_unparsed)} values:")
    print(still_unparsed['rating'].value_counts().head(10))

# Normalize ALL ratings to 0–1 for cross-dataset comparison
cleaned_goodreads_reviews['rating_normalized'] = cleaned_goodreads_reviews['rating'] / 5.0
cleaned_rotten_tomatoes_audience_reviews['rating_normalized'] = cleaned_rotten_tomatoes_audience_reviews['score'] / 5.0
cleaned_rotten_tomatoes_critic_reviews['rating_normalized'] = cleaned_rotten_tomatoes_critic_reviews['rating_parsed'] / 5.0

# Source labels for merged analysis
cleaned_goodreads_reviews['source'] = 'goodreads'
cleaned_rotten_tomatoes_audience_reviews['source'] = 'rt_audience'
cleaned_rotten_tomatoes_critic_reviews['source'] = 'rt_critic'

print("\nAll ratings normalized to 0–1:")
for label, df in [("Goodreads", cleaned_goodreads_reviews),
                  ("RT Audience", cleaned_rotten_tomatoes_audience_reviews),
                  ("RT Critics", cleaned_rotten_tomatoes_critic_reviews)]:
    col = df['rating_normalized'].dropna()
    print(f"  {label:14s}  range: {col.min():.2f} – {col.max():.2f}  mean: {col.mean():.3f}")

RT critic ratings: 25137 total, 9521 originally null
Parsed successfully: 15616/15616 non-null ratings (100.0%)

All ratings normalized to 0–1:
  Goodreads       range: 0.20 – 1.00  mean: 0.791
  RT Audience     range: 0.10 – 1.00  mean: 0.706
  RT Critics      range: 0.00 – 2.90  mean: 0.639


In [102]:
# normalizing titles
def normalize_title(title):
    """Lowercase, strip, collapse whitespace, remove leading articles."""
    if not isinstance(title, str):
        return ""
    title = title.lower().strip()
    title = re.sub(r'\s+', ' ', title)
    title = re.sub(r'^(the|a|an)\s+', '', title)
    return title

# Create join-friendly title columns
cleaned_goodreads_books['title_join'] = cleaned_goodreads_books['title'].apply(normalize_title)
cleaned_goodreads_reviews['title_join'] = cleaned_goodreads_reviews['book_title'].apply(normalize_title)
cleaned_rotten_tomatoes_movies['title_join'] = cleaned_rotten_tomatoes_movies['title'].apply(normalize_title)
cleaned_rotten_tomatoes_audience_reviews['title_join'] = cleaned_rotten_tomatoes_audience_reviews['movie_title'].apply(normalize_title)
cleaned_rotten_tomatoes_critic_reviews['title_join'] = cleaned_rotten_tomatoes_critic_reviews['movie_title'].apply(normalize_title)
cleaned_letterboxd_movies['title_join'] = cleaned_letterboxd_movies['Name'].apply(normalize_title)

# Report join coverage
rt_titles = set(cleaned_rotten_tomatoes_movies['title_join'])
gr_titles = set(cleaned_goodreads_books['title_join'])

for label, df, parent_titles in [
    ("RT audience → movies", cleaned_rotten_tomatoes_audience_reviews, rt_titles),
    ("RT critic  → movies", cleaned_rotten_tomatoes_critic_reviews, rt_titles),
    ("GR reviews → books",  cleaned_goodreads_reviews, gr_titles),
]:
    review_titles = set(df['title_join'])
    matched = len(review_titles & parent_titles)
    print(f"  {label}: {matched}/{len(review_titles)} titles match")


  RT audience → movies: 2867/2867 titles match
  RT critic  → movies: 2886/2886 titles match
  GR reviews → books: 5078/5078 titles match


In [ ]:
# parse the genres into the list
def parse_genre_string(genre_str):
    """Parse genre strings into clean Python lists."""
    if not isinstance(genre_str, str) or genre_str in ('Unknown', ''):
        return []
    # Pipe-separated (Goodreads & RT): "Fantasy|Fiction|Epic Fantasy"
    if '|' in genre_str:
        return [g.strip() for g in genre_str.split('|') if g.strip()]
    # Comma-separated (Letterboxd): "Drama, Comedy, Thriller"
    if ',' in genre_str:
        return [g.strip() for g in genre_str.split(',') if g.strip()]
    # Single genre with no delimiter
    return [genre_str.strip()]

cleaned_goodreads_books['genres_list'] = cleaned_goodreads_books['genres'].apply(parse_genre_string)
cleaned_rotten_tomatoes_movies['genres_list'] = cleaned_rotten_tomatoes_movies['genres'].apply(parse_genre_string)

# Top genres
for label, df in [("Goodreads", cleaned_goodreads_books), ("RT Movies", cleaned_rotten_tomatoes_movies)]:
    genre_counts = Counter(g for gl in df['genres_list'] for g in gl)
    print(f"\nTop 10 {label} genres:")
    for genre, count in genre_counts.most_common(10):
        print(f"  {genre}: {count}")



Top 10 Goodreads genres:
  Fiction: 2796
  Audiobook: 1795
  Fantasy: 1466
  Romance: 1274
  Nonfiction: 1034
  Young Adult: 949
  Classics: 869
  Contemporary: 843
  Adult: 810
  Book Club: 698

Top 10 RT Movies genres:
  Drama: 1185
  Comedy: 921
  Mystery & Thriller: 799
  Adventure: 642
  Action: 609
  Documentary: 404
  Fantasy: 378
  Horror: 374
  Animation: 374
  Crime: 360


In [ ]:
print("FINAL CLEANED DATASET SUMMARY")

datasets = {
    'Goodreads Books':      cleaned_goodreads_books,
    'Goodreads Reviews':    cleaned_goodreads_reviews,
    'RT Movies':            cleaned_rotten_tomatoes_movies,
    'RT Audience Reviews':  cleaned_rotten_tomatoes_audience_reviews,
    'RT Critic Reviews':    cleaned_rotten_tomatoes_critic_reviews,
}

FINAL CLEANED DATASET SUMMARY


**Saving Datasets**

In [ ]:
CLEAN_DIR = DATA_DIR + '/cleaned'
os.makedirs(CLEAN_DIR, exist_ok=True)

cleaned_goodreads_books.to_csv(CLEAN_DIR + '/cleaned_goodreads_books.csv', index=False)
cleaned_goodreads_reviews.to_csv(CLEAN_DIR + '/cleaned_goodreads_reviews.csv', index=False)
cleaned_rotten_tomatoes_movies.to_csv(CLEAN_DIR + '/cleaned_rotten_tomatoes_movies.csv', index=False)
cleaned_rotten_tomatoes_audience_reviews.to_csv(CLEAN_DIR + '/cleaned_rotten_tomatoes_audience_reviews.csv', index=False)
cleaned_rotten_tomatoes_critic_reviews.to_csv(CLEAN_DIR + '/cleaned_rotten_tomatoes_critic_reviews.csv', index=False)

print(f"All 6 cleaned datasets saved to: {CLEAN_DIR}/")
for f in sorted(os.listdir(CLEAN_DIR)):
    print(f"  {f}")

All 6 cleaned datasets saved to: data/cleaned/
  cleaned_goodreads_books.csv
  cleaned_goodreads_reviews.csv
  cleaned_letterboxd_movies.csv
  cleaned_rotten_tomatoes_audience_reviews.csv
  cleaned_rotten_tomatoes_critic_reviews.csv
  cleaned_rotten_tomatoes_movies.csv
